# Notebook 5: Evaluation & Decoding Strategies
Đánh giá độc lập 2 mô hình đã train (Full FT và LoRA) và tính điểm cuối cùng.
!pip install transformers datasets evaluate rouge_score bert_score peft tqdm
YÊU CẦU: BẬT GPU T4 x1 TRÊN KAGGLE

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel
from evaluate import load
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Tải test set
test_df = pd.read_csv("test_1k.csv")
articles = test_df["article"].tolist()
references = test_df["abstract"].tolist()

# Tải metrics
rouge = load("rouge")
# bertscore = load("bertscore") # Bật dòng này nếu bạn muốn chạy BERTScore, quá trình tính toán khá tốn RAM

In [ ]:
# 1. Hàm Inference đa cấu hình
def generate_summaries(model, tokenizer, texts, decoding_strategy="beam"):
    summaries = []
    # Chỉ chạy 100 mẫu demo để notebook chạy nhanh (chỉnh range thành len(texts) cho toàn bộ)
    eval_texts = texts[:100] 
    
    for text in tqdm(eval_texts, desc=f"Generating ({decoding_strategy})"):
        inputs = tokenizer(text, max_length=512, truncation=True, return_tensors="pt").to(device)
        
        if decoding_strategy == "beam":
            outputs = model.generate(**inputs, max_length=128, num_beams=4, early_stopping=True)
        elif decoding_strategy == "sampling_top_p":
            outputs = model.generate(**inputs, max_length=128, do_sample=True, top_p=0.9, top_k=50)
            
        summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
        summaries.append(summary)
    return summaries, eval_texts

def evaluate_summaries(predictions, refs):
    # Cắt reference khớp với số prediction đã sinh ra
    actual_refs = refs[:len(predictions)]
    r_score = rouge.compute(predictions=predictions, references=actual_refs)
    
    # Bỏ comment 3 dòng dưới để tính BERTScore
    # b_score = bertscore.compute(predictions=predictions, references=actual_refs, lang="vi")
    # b_f1 = sum(b_score['f1']) / len(b_score['f1'])
    # return {"ROUGE-1": r_score['rouge1'], "ROUGE-L": r_score['rougeL'], "BERTScore_F1": b_f1}
    
    return {"ROUGE-1": r_score['rouge1'], "ROUGE-L": r_score['rougeL']}

In [ ]:
# 2. Đánh giá Full FT Model (Beam Search vs Top-p)
print("\n--- ĐÁNH GIÁ MÔ HÌNH FULL FT ---")
# Thay đường dẫn này bằng folder chứa model từ NB3 (hoặc Dataset gắn trên Kaggle)
FULL_FT_PATH = "./bartpho_full_ft_final" 
try:
    tokenizer_full = AutoTokenizer.from_pretrained(FULL_FT_PATH)
    model_full = AutoModelForSeq2SeqLM.from_pretrained(FULL_FT_PATH).to(device)

    print(">> Full FT - Beam Search...")
    preds_full_beam, _ = generate_summaries(model_full, tokenizer_full, articles, "beam")
    print("KQ Full FT (Beam):", evaluate_summaries(preds_full_beam, references))

    print(">> Full FT - Top-p Sampling...")
    preds_full_topp, _ = generate_summaries(model_full, tokenizer_full, articles, "sampling_top_p")
    print("KQ Full FT (Top-p):", evaluate_summaries(preds_full_topp, references))

    del model_full
    torch.cuda.empty_cache()
except Exception as e:
    print("Lỗi load Full FT (Có thể bạn chưa gắn Dataset model vào Kaggle):", e)

In [ ]:
# 3. Đánh giá LoRA Model (Beam Search)
print("\n--- ĐÁNH GIÁ MÔ HÌNH LORA ---")
# Thay đường dẫn này bằng folder chứa adapter LoRA từ NB4
LORA_PATH = "./bartpho_lora_final"
BASE_MODEL_NAME = "vinai/bartpho-syllable"
try:
    print("Loading Base Model for LoRA...")
    base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL_NAME)
    model_lora = PeftModel.from_pretrained(base_model, LORA_PATH).to(device)
    tokenizer_lora = AutoTokenizer.from_pretrained(LORA_PATH)

    print(">> LoRA - Beam Search...")
    preds_lora_beam, final_texts = generate_summaries(model_lora, tokenizer_lora, articles, "beam")
    print("KQ LoRA (Beam):", evaluate_summaries(preds_lora_beam, references))
except Exception as e:
    print("Lỗi load LoRA (Có thể bạn chưa gắn Dataset adapter vào Kaggle):", e)

In [ ]:
# 4. Phục vụ Error Analysis (Xuất ra CSV)
# Lưu ý: Code này giả sử bạn đã chạy thành công cả Full FT và LoRA. 
# Cần format lại dataframe tùy thuộc vào việc mô hình nào load thành công.
print("Phần xuất kết quả ra file csv để kiểm tra thủ công...")